# Anomaly Detection System - Exploratory Analysis

This notebook provides interactive exploration of the anomaly detection system.

**Author:** Your Name  
**Date:** January 2026

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from anomaly_detector import SecureAnomalyDetector, generate_synthetic_dataset

%matplotlib inline
sns.set_style('whitegrid')

## 1. Data Generation and Exploration

In [ ]:
# Generate synthetic dataset
X, y = generate_synthetic_dataset(n_samples=5000, n_features=10, contamination=0.1)

print(f"Dataset shape: {X.shape}")
print(f"Normal samples: {(y == 1).sum()}")
print(f"Anomalies: {(y == -1).sum()}")
print(f"\nFirst few rows:")
X.head()

## 2. Data Visualization

In [ ]:
# Visualize first two features
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X.iloc[:, 0][y == 1], X.iloc[:, 1][y == 1], 
           alpha=0.5, label='Normal', s=20)
plt.scatter(X.iloc[:, 0][y == -1], X.iloc[:, 1][y == -1], 
           alpha=0.7, label='Anomaly', s=50, marker='x', color='red')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Data Distribution (2D Projection)')
plt.legend()

plt.subplot(1, 2, 2)
X.iloc[:, :5].boxplot()
plt.title('Feature Distribution')
plt.ylabel('Value')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 3. Model Training and Prediction

In [ ]:
# Initialize and train detector
from sklearn.model_selection import train_test_split

detector = SecureAnomalyDetector(contamination=0.1, random_state=42)

# Preprocess
X_processed = detector.preprocess_data(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.3, random_state=42, stratify=y
)

# Train
detector.train(X_train.values)
print("✓ Model trained successfully")

## 4. Model Evaluation

In [ ]:
# Evaluate model
metrics = detector.evaluate(X_test.values, y_test)

print(f"\nModel Performance:")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall: {metrics['recall']:.4f}")
print(f"  F1-Score: {metrics['f1_score']:.4f}")

## 5. Interactive Analysis

In [ ]:
# Get predictions and scores
predictions, scores = detector.predict(X_test.values)

# Create analysis dataframe
analysis_df = pd.DataFrame({
    'true_label': y_test,
    'predicted_label': predictions,
    'anomaly_score': scores,
    'correct': y_test == predictions
})

print("Sample predictions:")
analysis_df.head(10)

In [ ]:
# Visualize anomaly scores
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(scores[y_test == 1], bins=30, alpha=0.7, label='Normal', color='blue')
plt.hist(scores[y_test == -1], bins=30, alpha=0.7, label='Anomaly', color='red')
plt.xlabel('Anomaly Score')
plt.ylabel('Frequency')
plt.title('Anomaly Score Distribution by True Label')
plt.legend()

plt.subplot(1, 2, 2)
sns.boxplot(data=analysis_df, x='true_label', y='anomaly_score')
plt.xlabel('True Label (1=Normal, -1=Anomaly)')
plt.ylabel('Anomaly Score')
plt.title('Anomaly Score by True Label')

plt.tight_layout()
plt.show()

## 6. Error Analysis

In [ ]:
# Find false positives and false negatives
false_positives = analysis_df[(analysis_df['true_label'] == 1) & (analysis_df['predicted_label'] == -1)]
false_negatives = analysis_df[(analysis_df['true_label'] == -1) & (analysis_df['predicted_label'] == 1)]

print(f"False Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}")
print(f"\nFalse Positive Rate: {len(false_positives) / (y_test == 1).sum():.2%}")
print(f"False Negative Rate: {len(false_negatives) / (y_test == -1).sum():.2%}")

## 7. Conclusion

This notebook demonstrated:
- Data generation and exploration
- Model training and evaluation
- Interactive visualization of results
- Error analysis

Next steps:
- Try different contamination values
- Experiment with feature engineering
- Compare with other algorithms (One-Class SVM, LOF)
- Test on real-world datasets